# 合并结肠组织的预处理数据（非药物处理样本）

# 文件说明
- **文件名**: tissue_dataset_merged.h5ad 
- **存放位置**: 坚果云  `UC/scRNA-seq_data/04_dataset_merge/`  
- **生成日期**: 2025-11-9  
- **生成者**: Jiang Lai
- **对应脚本**: notebooks/1_preprocessing/1.3_data_merge/1_tissue_dataset_merge.ipynb  
- **内容说明**: 合并所有非药物处理的结肠组织数据  

## 环境与路径

In [1]:
# === 基础依赖 ===
import os
import re
import scanpy as sc
import pandas as pd
from typing import List

# === 数据路径（按你的目录） ===
DATA_DIR = "/mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/03_qc_filtered"
SAVE_DIR = "/mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/04_dataset_merge"
SAVE_PATH = os.path.join(SAVE_DIR, "tissue_dataset_merged.h5ad")

os.makedirs(SAVE_DIR, exist_ok=True)

# === 全局参数（可一键调整） ===
MIN_FRACTION = 0.80  # 至少出现在 80% 数据集（你可改成 0.7 / 0.9）
EXCLUDE_KEYWORD = "pbmc"  # 排除 pbmc
PRINT_TOP_N = 5           # 打印前 N 个日志预览

/home/future/miniconda3/envs/omicverse/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## 1.收集文件与读取（排除PBMC样本）

In [2]:
# 收集所有 h5ad 文件，排除含有 "pbmc" 的数据（大小写不敏感）
files = sorted([
    f for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".h5ad") and EXCLUDE_KEYWORD.lower() not in f.lower()
])

assert len(files) > 0, "未在目录中找到 h5ad 文件（或全被排除）。"

print(f"将要合并的结肠组织数据集文件（{len(files)} 个）：")
for f in files[:PRINT_TOP_N]:
    print("  -", f)
if len(files) > PRINT_TOP_N:
    print(f"  ... 共 {len(files)} 个\n")

将要合并的结肠组织数据集文件（11 个）：
  - 10_GSE296969_qc.h5ad
  - 11_GSE235663_qc.h5ad
  - 1_GSE182270_qc.h5ad
  - 2_GSE214695_qc.h5ad
  - 3_GSE150115_qc.h5ad
  ... 共 11 个



## 2.提取数据集编号并读取数据（GSE/SCP → obs['GSE']）

In [4]:
def extract_dataset_id(fname: str) -> str:
    """从文件名中提取数据集编号（GSE 或 SCP），若无则使用去后缀的文件名。"""
    m = re.search(r'(GSE\d+|SCP\d+)', fname, flags=re.IGNORECASE)
    return m.group(1).upper() if m else os.path.splitext(fname)[0]

adatas: List[sc.AnnData] = []
for f in files:
    path = os.path.join(DATA_DIR, f)
    ad = sc.read_h5ad(path)
    ds_id = extract_dataset_id(f)  # 如 GSE125527 或 SCP259
    ad.obs["GSE"] = ds_id

    # 节省内存：把 GSE 转为分类类型（不会影响后续）
    ad.obs["GSE"] = ad.obs["GSE"].astype("category")

    adatas.append(ad)
    print(f"Loaded {f:<30} -> GSE={ds_id:<10} | cells={ad.n_obs:,}")

print(f"\n共读取 {len(adatas)} 个 AnnData 对象。")

Loaded 10_GSE296969_qc.h5ad           -> GSE=GSE296969  | cells=98,112
Loaded 11_GSE235663_qc.h5ad           -> GSE=GSE235663  | cells=11,042
Loaded 1_GSE182270_qc.h5ad            -> GSE=GSE182270  | cells=26,128
Loaded 2_GSE214695_qc.h5ad            -> GSE=GSE214695  | cells=8,908
Loaded 3_GSE150115_qc.h5ad            -> GSE=GSE150115  | cells=7,230
Loaded 4_GSE116222_qc.h5ad            -> GSE=GSE116222  | cells=11,113
Loaded 5_GSE134649_qc.h5ad            -> GSE=GSE134649  | cells=10,272
Loaded 6_GSE125527_tissue_qc.h5ad     -> GSE=GSE125527  | cells=43,241
Loaded 7_GSE231993_qc.h5ad            -> GSE=GSE231993  | cells=36,404
Loaded 8_SCP259_qc.h5ad               -> GSE=SCP259     | cells=113,095
Loaded 9_GSE114374_qc.h5ad            -> GSE=GSE114374  | cells=8,962

共读取 11 个 AnnData 对象。


## 3.统计各数据集的基因病选择“≥80%”的保留集合
原理：对每个基因统计它出现在多少个数据集的 var_names 中，保留出现频次 ≥ MIN_FRACTION × 数据集数量的基因。  
这样比“纯交集（100%）”更稳，不会因为个别数据集的缺失把大量有用基因排掉。

In [5]:
# 获取每个数据集的基因集合
gene_sets = [set(ad_.var_names) for ad_ in adatas]
all_genes = set().union(*gene_sets)

# 统计每个基因出现在哪些数据集
gene_counts = {g: sum(g in s for s in gene_sets) for g in all_genes}

# 确定阈值
min_datasets = int(len(adatas) * MIN_FRACTION)
min_datasets = max(min_datasets, 1)

common_genes = sorted([g for g, c in gene_counts.items() if c >= min_datasets])

print(f"基因总数：{len(all_genes):,}，保留 {len(common_genes):,} 个（出现在 ≥{min_datasets}/{len(adatas)} 个数据集中）")
assert len(common_genes) > 0, "没有基因满足阈值，请降低 MIN_FRACTION。"


基因总数：38,482，保留 11,743 个（出现在 ≥8/11 个数据集中）


## 4. 外连接合并

In [6]:
# 外连接合并所有数据集，不生成 batch 列
adata_merged = sc.concat(
    adatas,
    join="outer",       # 外连接：保留所有基因
    label=None,         # 不生成 batch 列
    axis=0,
    index_unique=None   # 保留原 obs_names
)

print(f"[outer-merge] 合并后维度：cells={adata_merged.n_obs:,}, genes={adata_merged.n_vars:,}")

# 按保留的基因集合裁剪
adata_merged = adata_merged[:, common_genes].copy()
print(f"[subset to >=80% genes] 维度：cells={adata_merged.n_obs:,}, genes={adata_merged.n_vars:,}")

/home/future/miniconda3/envs/omicverse/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


[outer-merge] 合并后维度：cells=374,507, genes=38,482
[subset to >=80% genes] 维度：cells=374,507, genes=11,743


## 5. 只保留 var['mt'] 一列

In [7]:
if "mt" not in adata_merged.var.columns:
    adata_merged.var["mt"] = adata_merged.var_names.str.upper().str.startswith("MT-")
else:
    adata_merged.var["mt"] = adata_merged.var["mt"].astype(bool)

adata_merged.var = adata_merged.var.loc[:, ["mt"]]
print(f"var 列：{list(adata_merged.var.columns)}")


var 列：['mt']


## 6.检查合并结果（GSE 分布、var 预览）

In [9]:
vc = adata_merged.obs["GSE"].value_counts()
print("GSE 分布（前 10 项）：")
print(vc.head(10))
print(f"... 共 {vc.shape[0]} 个数据集\n")

GSE 分布（前 10 项）：
GSE
SCP259       113095
GSE296969     98112
GSE125527     43241
GSE231993     36404
GSE182270     26128
GSE116222     11113
GSE235663     11042
GSE134649     10272
GSE114374      8962
GSE214695      8908
Name: count, dtype: int64
... 共 11 个数据集



## 7.保存

In [10]:
adata_merged.write_h5ad(SAVE_PATH, compression="gzip")
print(f"✅ 合并后的数据已保存到：{SAVE_PATH}")

✅ 合并后的数据已保存到：/mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/04_dataset_merge/tissue_dataset_merged.h5ad
